# Complaint Triage with LiteLLM

[LiteLLM](https://docs.litellm.ai/) gives a single, OpenAI-style interface (`completion`) to many providers. Here we run the same classification task against **either** a local Ollama model **or** a hosted Groq model, switched by a single variable.

In [1]:
!pip install litellm


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Select the provider

Set `PROVIDER` to either `"ollama"` (local) or `"groq"` (hosted). LiteLLM identifies the backend from the `provider/model` prefix in the model name.

For Groq, set your API key: `export GROQ_API_KEY=...` before launching the notebook.

In [5]:
import os
from getpass import getpass

# Switch between "ollama" and "groq"
PROVIDER = "groq"

if PROVIDER == "ollama":
    MODEL = "ollama/gemma3:1b"
    API_BASE = "http://localhost:11434"
elif PROVIDER == "groq":
    MODEL = "groq/llama-3.3-70b-versatile"
    API_BASE = None  # uses Groq's hosted endpoint
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")  # or set it in your shell
else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

print(f"Using provider={PROVIDER}, model={MODEL}")

Using provider=groq, model=groq/llama-3.3-70b-versatile


## Classification task

In [6]:
from litellm import completion

complaint = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

categories = ['Delivery issue', 'Product quality', 'Customer service', 'Other']

system_instruction = (
    'You are a customer-support triage assistant. '
    'Classify each complaint into exactly one of the following categories: '
    f'{", ".join(categories)}. '
    'Respond with only the category name — no explanation, no punctuation.'
)

user_message = f'Complaint: {complaint}'

print('========== Prompt (system) ==========')
print(system_instruction)
print('\n========== Prompt (user) ==========')
print(user_message)

response = completion(
    model=MODEL,
    api_base=API_BASE,
    messages=[
        {'role': 'system', 'content': system_instruction},
        {'role': 'user',   'content': user_message},
    ],
)

print('\n========== Response ==========')
print(response.choices[0].message.content)

========== Prompt (system) ==========
You are a customer-support triage assistant. Classify each complaint into exactly one of the following categories: Delivery issue, Product quality, Customer service, Other. Respond with only the category name — no explanation, no punctuation.

========== Prompt (user) ==========
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

========== Response ==========
Delivery issue
